# Домашняя работа 2 — качество данных

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 3.1.0


## Независимая загрузка

In [2]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)
print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

(6830, 7) (6832, 7)
source_row       string
activity_year    string
lei              string
county_code      string
action_taken     string
loan_amount      string
income           string
dtype: object


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0
2,5,2023,549300P1347BFCBHJV60,11001,1,275000.0,BAD
3,7,2023,549300RPOGWJRH63HS39,11001,1,295000.0,63.0
4,8,2023,549300RPOGWJRH63HS39,11001,1,1125000.0,247.0


## H1. Собственная функция очистки

In [3]:

def clean_export(source):
    import pandas as pd

    original = source.copy(deep=True).reset_index(drop=True)
    columns = list(original.columns)

    df = original.copy()
    df["_row_id"] = range(len(df))

    known = (
        df["source_row"].notna()
        & df["source_row"].str.strip().ne("").fillna(False)
    )

    for source_row, group in df.loc[known].groupby("source_row"):
        if len(group[columns].drop_duplicates()) > 1:
            raise AssertionError(
                f"Конфликтующие версии source_row={source_row}"
            )

    input_count = len(df)
    df = df.drop_duplicates(subset=columns, keep="first").copy()
    replays = input_count - len(df)

    df["county_code"] = df["county_code"].str.strip()

    parse_errors = 0

    for col in [
        "source_row",
        "activity_year",
        "action_taken",
        "loan_amount",
        "income",
    ]:
        values = df[col].str.strip().replace("", pd.NA)
        numbers = pd.to_numeric(values, errors="coerce")

        parse_errors += int(
            (values.notna() & numbers.isna()).sum()
        )
        df[col] = numbers

    valid = (
        df["source_row"].notna()
        & df["action_taken"].isin([1, 2, 3])
        & df["loan_amount"].gt(0)
        & df["county_code"].eq("11001")
    ).fillna(False)

    clean = df.loc[valid, columns].copy()

    clean["activity_year"] = clean["activity_year"].map(
        lambda x: (
            str(int(x))
            if pd.notna(x) and float(x).is_integer()
            else pd.NA
        )
    ).astype("string")

    bad = df.loc[~valid].copy()

    quarantine = original.iloc[
        bad["_row_id"].astype(int).to_numpy()
    ].copy()

    reasons = []
    for _, row in bad.iterrows():
        row_reasons = []

        if pd.isna(row["source_row"]):
            row_reasons.append("unknown_source_row")
        if pd.isna(row["action_taken"]) or row["action_taken"] not in [1, 2, 3]:
            row_reasons.append("invalid_action_taken")

        if pd.isna(row["loan_amount"]):
            row_reasons.append("missing_or_invalid_loan_amount")
        elif row["loan_amount"] <= 0:
            row_reasons.append("nonpositive_loan_amount")

        if pd.isna(row["county_code"]) or row["county_code"] != "11001":
            row_reasons.append("invalid_county_code")

        reasons.append(
            "; ".join(row_reasons)
            if row_reasons
            else "invalid_record"
        )

    quarantine["reason"] = reasons
    audit = {
        "input": input_count,
        "replays": replays,
        "clean": len(clean),
        "quarantine": len(quarantine),
        "parse_errors": parse_errors,
    }
    assert input_count == (
        audit["replays"] + audit["clean"] + audit["quarantine"]
    ), "Ошибка баланса строк"

    return clean, quarantine, audit


## H2. Год и известный доход

In [4]:
clean, quarantine, audit = clean_export(export)

print("Журнал:", audit)
print("Баланс:", audit["input"], "=", audit["replays"], "+",
      audit["clean"], "+", audit["quarantine"])

year_ok = clean["activity_year"].eq("2023").fillna(False).all()
print("Все подготовленные записи за 2023 год:", year_ok)

known_income = clean["income"].dropna()
print("Всего подготовленных записей:", len(clean))
print("Известный доход:", int(clean["income"].notna().sum()))
print("Неизвестный доход:", int(clean["income"].isna().sum()))
print("Средний известный доход:", known_income.mean(), "тыс. USD в год")

assert year_ok
assert audit["input"] == audit["replays"] + audit["clean"] + audit["quarantine"]
assert_frame_equal(export, raw_copy)


Журнал: {'input': 6832, 'replays': 2, 'clean': 6828, 'quarantine': 2, 'parse_errors': 2}
Баланс: 6832 = 2 + 6828 + 2
Все подготовленные записи за 2023 год: True
Всего подготовленных записей: 6828
Известный доход: 6485
Неизвестный доход: 343
Средний известный доход: 237.06553585196608 тыс. USD в год


## H3. Проверка отчётного года

In [5]:
wrong_year = export.copy(deep=True)
wrong_year.loc[wrong_year.index[0], "activity_year"] = "2022"

year_ok = wrong_year["activity_year"].eq("2023").fillna(False).all()
print("Проверка года пройдена:", year_ok)

assert not year_ok, "Проверка не обнаружила неверный год"
assert_frame_equal(export, raw_copy)


Проверка года пройдена: False


## H+1. Конфликт исходных значений

## H+2. Совпадение опубликованных полей